# CausalEPP: bounded-memory LLM scoring

**Start with Runtime → Restart session**, select a GPU, then run from the top. The
old crash may leave live tensors referenced by the notebook traceback; empty_cache
cannot free those. This notebook generates matching scores, not training.

Changes: FP16 weights, explicit non-thinking template, bounded prompt/candidate/output
sizes, automatic OOM splitting, per-chunk checkpoints, and honest incomplete-cache
validation. Candidate IDs and training-only history are preserved. Chunking changes
the context seen by the LLM and can change scores, so **all parts must use this version**
and its separate cache directory. Do not merge these scores with continuous_v2.

First run uses two chunks as a smoke check. If successful, set `MAX_JOBS=0` and rerun
the partition cell to finish. No T4 execution was performed by the author of this fix;
local tests verify scheduling/recovery behavior, not GPU capacity or score quality.


In [ ]:
# Run in a fresh session; restart again if Colab requests it after installation.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
%pip install -q "transformers==4.57.1" "huggingface-hub>=0.34,<1" "accelerate>=0.26,<2" pandas numpy sentencepiece
# Keep the existing project if already present. This does not pull/overwrite local changes.
from pathlib import Path
if not Path('/content/CausalEPP').exists():
    !git clone -q --depth 1 --branch calibrationBranch https://github.com/karankashyap10/Popularity-Debiasing-Causal-EPP.git /content/CausalEPP


## Account configuration
**Set the correct `PART_ID`** for each account. The split uses sorted *actual user IDs*, not assumptions that IDs are sequential. The full train/validation/test data must be identical on all accounts.


In [ ]:
PART_ID = 1
TOTAL_PARTS = 10
MODEL_ID = "Qwen/Qwen3-0.6B"
MODEL_REVISION = "c1899de289a04d12100db370d81485cdf75e47ca"
CANDIDATE_POOL = 50  # minimum; every original training-positive candidate is retained
HISTORY_LENGTH = 20
BATCH_SIZE = 2  # one setting only; 4/8/16 are upper bounds, automatic OOM splitting applies
CANDIDATES_PER_PROMPT = 16
MAX_INPUT_TOKENS = 1024
MAX_NEW_TOKENS = 384
MAX_JOBS = 2  # initial smoke check; change to 0, rerun partition cell for the full job
PROMPT_VERSION = "continuous_v3_chunked_no_think"
DATA_DIR = "/content/CausalEPP/data/Ciao"
SAVE_TO_DRIVE = True
assert 1 <= PART_ID <= TOTAL_PARTS
assert BATCH_SIZE >= 1 and CANDIDATES_PER_PROMPT >= 1 and MAX_JOBS >= 0


In [ ]:
from pathlib import Path
import json, hashlib, os, re, gc
import pandas as pd
import numpy as np
import torch
from tqdm.auto import tqdm

print("GPU available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in Colab: Runtime > Change runtime type > GPU")
print("GPU:", torch.cuda.get_device_name(0))
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = Path('/content/drive/MyDrive/CausalEPP_llm_cache')
else:
    OUT_DIR = Path('/content/llm_cache')
OUT_DIR = OUT_DIR / PROMPT_VERSION
OUT_DIR.mkdir(parents=True, exist_ok=True)


## Validate the input files
**Only these four CSV files are needed to generate matching scores.** The processed temporal `.npy` files are required later for full CausalEPP training but not this cache-generation job.


In [ ]:
data_dir = Path(DATA_DIR)
required = ['train_data.csv', 'val_data.csv', 'test_data.csv', 'item_mapping.csv']
missing = [f for f in required if not (data_dir/f).is_file() or (data_dir/f).stat().st_size == 0]
if missing:
    raise FileNotFoundError(f"Missing or empty files in {DATA_DIR}: {missing}. Upload the processed files before proceeding.")
train = pd.read_csv(data_dir/'train_data.csv', sep='\t')
val = pd.read_csv(data_dir/'val_data.csv', sep='\t')
test = pd.read_csv(data_dir/'test_data.csv', sep='\t')
item_mapping = pd.read_csv(data_dir/'item_mapping.csv')
for name,df in [('train',train),('val',val),('test',test)]:
    assert {'user','item','timestamp'}.issubset(df.columns), (name,list(df.columns))
assert {'item','raw_item'}.issubset(item_mapping.columns)
all_users = sorted(set(pd.concat([train.user, val.user, test.user]).astype(int)))
print('Interactions:',len(train),len(val),len(test),'Unique users:',len(all_users))

# Adjust expectations dynamically instead of failing due to minor dataset version changes
EXPECTED_USERS = len(all_users)
parts = np.array_split(np.array(all_users, dtype=int), TOTAL_PARTS)
selected_users = [int(u) for u in parts[PART_ID-1]]

EXPECTED_PER_PART = len(selected_users)
assert sorted([u for p in parts for u in p]) == all_users
print(f'PART {PART_ID}: {len(selected_users)} users')
print('Sample:',selected_users[:8])

## Build identical candidate pools as `llm_baseline.py`
Candidates comprise all training positives plus globally popular training items until the pool target is met. Where necessary, one unseen candidate is added. User profiles use training interactions only, sorted chronologically.


In [ ]:
item_names = dict(zip(item_mapping['item'].astype(int), item_mapping['raw_item'].astype(str)))
num_items = max(int(x.item.max()) for x in (train,val,test) if not x.empty)+1
counts = train['item'].value_counts().reindex(range(num_items),fill_value=0)
popular_items = counts.sort_values(ascending=False).index.tolist()
histories = {int(u): rows.sort_values('timestamp')['item'].astype(int).tolist() for u,rows in train.groupby('user')}
requests = {}
for user in selected_users:
    history = histories.get(user, [])
    positives = set(history)
    candidates = set(positives)
    for item in popular_items:
        candidates.add(int(item))
        if len(candidates) >= CANDIDATE_POOL:
            break
    if positives and not any(i not in positives for i in candidates):
        neg = next((int(i) for i in popular_items if int(i) not in positives), None)
        if neg is not None: candidates.add(neg)
    candidate_ids = sorted(candidates)
    profile = '\n'.join(item_names.get(i,f'Item {i}') for i in history[-HISTORY_LENGTH:]) or 'No prior item history.'
    requests[user] = (profile, [(i,item_names.get(i,f'Item {i}')) for i in candidate_ids])
print('Candidate item counts:', pd.Series({u:len(x[1]) for u,x in requests.items()}).describe())


## Reproducibility fingerprint and resumable output
Every part contains `{"user_id": {"item_id": score}}`. A separate manifest records the model, input hashes, prompt version, assigned users, and settings. Output is saved after every scored batch.


In [ ]:
def file_sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()
settings = dict(model_id=MODEL_ID, candidate_pool=CANDIDATE_POOL, history_length=HISTORY_LENGTH,
                prompt_version=PROMPT_VERSION, model_revision=MODEL_REVISION,
                candidates_per_prompt=CANDIDATES_PER_PROMPT, max_input_tokens=MAX_INPUT_TOKENS,
                max_new_tokens=MAX_NEW_TOKENS, dtype="float16", enable_thinking=False, num_items=num_items,
                data_sha256={p:file_sha256(data_dir/p) for p in required})
fingerprint = hashlib.sha256(json.dumps(settings,sort_keys=True).encode()).hexdigest()
cache_file = OUT_DIR / f'llm_cache_part_{PART_ID:02d}.json'
manifest_file = OUT_DIR / f'llm_cache_part_{PART_ID:02d}_manifest.json'
manifest = dict(part_id=PART_ID, total_parts=TOTAL_PARTS, users=selected_users,
                settings=settings, fingerprint=fingerprint)
if cache_file.exists() and not manifest_file.exists():
    raise RuntimeError("Cache has no manifest; move it aside rather than mixing unknown scores.")
if manifest_file.exists():
    old=json.loads(manifest_file.read_text())
    if old != manifest:
        raise RuntimeError('Existing manifest differs: check PART_ID, inputs or settings; do not overwrite inconsistent cache.')
manifest_file.write_text(json.dumps(manifest,indent=2))
cache = json.loads(cache_file.read_text()) if cache_file.exists() else {}
assert set(cache).issubset(set(map(str,selected_users))), 'Cache contains out-of-partition users'
print('Already cached users:', len(cache), 'Fingerprint:',fingerprint[:16])

def atomic_write(path,obj):
    tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj,ensure_ascii=False))
    os.replace(tmp,path)


## Load once, then bounded generation
FP16 is supported on T4. The original traceback already used SDPA; switching to SDPA
alone is not the fix. The bounds below control actual request sizes. No automatic
truncation of titles, profiles or candidate IDs is performed. An oversized single-item
prompt produces a descriptive error. Explicitly shorten HISTORY_LENGTH or inspect
malformed titles and start a new consistent cache if that happens.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
# Restart the session first after a prior OOM; do not repeatedly load duplicate models.
if 'model' in globals():
    del model
    gc.collect()
    torch.cuda.empty_cache()
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
tokenizer.padding_side='left'
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, torch_dtype=torch.float16,
    device_map={'':0}, attn_implementation='sdpa', low_cpu_mem_usage=True)
model.eval()
DEVICE=next(model.parameters()).device
print('Model device/dtype:', DEVICE, next(model.parameters()).dtype)
print('Free / total GPU GiB:', tuple(round(x/2**30,2) for x in torch.cuda.mem_get_info()))

def parse_scores(text, candidate_ids):
    match=re.search(r'\{[\s\S]*\}',text)
    raw={}
    if match:
        try:
            obj=json.loads(match.group(0))
            if isinstance(obj,dict):raw=obj
        except json.JSONDecodeError:pass
    if not raw:
        pairs=re.findall(r'["\']?(\d+)["\']?\s*:\s*(-?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?)',text)
        raw=dict(pairs)
    scores={}
    for item in candidate_ids:
        value=raw.get(str(item),raw.get(item))
        if value is None: continue
        try:value=float(value)
        except (ValueError,TypeError):continue
        if np.isfinite(value):scores[int(item)]=float(np.clip(value,0.0,1.0))
    return scores

def format_prompt(profile, candidates):
    req={"profile":profile,"pending":candidates}
    candidate_text='\n'.join(f"{i}\t{name.replace(chr(9),' ').replace(chr(10),' ')}" for i,name in req['pending'])
    prompt=(
        "Estimate a continuous relevance score for every candidate item based only on "
        "the user's previously interacted item titles. Each score must be a JSON number "
        "between 0.0 and 1.0 inclusive. Use the entire continuous range, including decimal "
        "values between the endpoints (e.g., 0.13, 0.47, 0.86) whenever appropriate; "
        "do not restrict scores to 0, 0.5, or 1. A higher score means greater relevance. "
        "Return exactly one score for EVERY candidate ID. Output only a JSON object "
        "with item IDs as keys and numeric scores as values, without explanation. /no_think\n\n"
        f"User's past item titles, oldest first:\n{req['profile']}\n\n"
        f"Candidates, ID then title:\n{candidate_text}"
    )
    return tokenizer.apply_chat_template([{"role":"user","content":prompt}], add_generation_prompt=True, tokenize=False, enable_thinking=False)

def prompt_tokens(profile, candidates):
    return len(tokenizer(format_prompt(profile,candidates),add_special_tokens=False)['input_ids'])

def bounded_jobs(user, profile, candidates):
    # Bound both candidate count AND actual tokenizer length; never discard an item.
    for start in range(0,len(candidates),CANDIDATES_PER_PROMPT):
        yield from split_to_fit(user,profile,candidates[start:start+CANDIDATES_PER_PROMPT])

def split_to_fit(user,profile,candidates):
    if not candidates:return
    if prompt_tokens(profile,candidates)<=MAX_INPUT_TOKENS:
        yield (user,profile,candidates)
    elif len(candidates)>1:
        mid=len(candidates)//2
        yield from split_to_fit(user,profile,candidates[:mid])
        yield from split_to_fit(user,profile,candidates[mid:])
    else:
        raise ValueError(f'User {user}, item {candidates[0][0]} exceeds the input-token limit even alone. '
                         'Inspect unusually long titles/history; no content was silently truncated.')

def generate_once(jobs):
    inputs=None; output=None
    try:
        prompts=[format_prompt(p,c) for _,p,c in jobs]
        inputs=tokenizer(prompts,return_tensors='pt',padding=True,add_special_tokens=False)
        width=inputs['input_ids'].shape[-1]
        if width>MAX_INPUT_TOKENS:raise ValueError('Prompt exceeds token budget.')
        limit=min(MAX_NEW_TOKENS,32+20*max(len(c) for _,_,c in jobs))
        if width+limit>model.config.max_position_embeddings:
            raise ValueError('Prompt and output exceed model context limit.')
        inputs={k:v.to(DEVICE) for k,v in inputs.items()}
        with torch.inference_mode():
            output=model.generate(**inputs,max_new_tokens=limit,do_sample=False,
                use_cache=True,pad_token_id=tokenizer.pad_token_id,
                output_scores=False,output_attentions=False,output_hidden_states=False,
                return_dict_in_generate=False)
        return tokenizer.batch_decode(output[:,width:].cpu(),skip_special_tokens=True)
    finally:
        # Release both successful and failed-call tensors before an outer retry.
        del inputs, output

def score_jobs(jobs):
    pending=[{'user':u,'profile':p,'pending':list(c),'scores':{}} for u,p,c in jobs]
    for attempt in range(3):
        active=[r for r in pending if r['pending']]
        if not active:break
        responses=generate_once([(r['user'],r['profile'],r['pending']) for r in active])
        for r,response in zip(active,responses):
            r['scores'].update(parse_scores(response,[i for i,_ in r['pending']]))
            r['pending']=[pair for pair in r['pending'] if pair[0] not in r['scores']]
    results=[]
    for r in pending:
        fallback=[i for i,_ in r['pending']]
        r['scores'].update({i:0.5 for i in fallback})  # existing policy, explicitly recorded
        results.append((r['user'],r['scores'],fallback))
    return results

def adaptive_results(jobs):
    # Catch OOM only. Other errors must remain visible. Leave the except block
    # before retrying so its traceback does not retain GPU tensors.
    failed=False
    try:
        result=score_jobs(jobs)
    except torch.cuda.OutOfMemoryError:
        failed=True
    if not failed:
        yield from result
        return
    gc.collect();torch.cuda.empty_cache()
    print('CUDA OOM: reducing work size; completed chunks stay on disk.')
    if len(jobs)>1:
        middle=len(jobs)//2
        yield from adaptive_results(jobs[:middle])
        yield from adaptive_results(jobs[middle:])
    else:
        u,p,c=jobs[0]
        if len(c)<=1:
            raise RuntimeError('GPU OOM even for one item. Restart the Colab session and inspect '
                               'GPU memory before loading; no more automatic retries. Cache is preserved.')
        middle=len(c)//2
        yield from adaptive_results([(u,p,c[:middle])])
        yield from adaptive_results([(u,p,c[middle:])])


## Score assigned partition, checkpointing each completed chunk
`BATCH_SIZE` is the maximum number of small prompts submitted together, not the
number of unbounded user histories. The scheduler sorts by token length to reduce
padding. OOM retries split batches, then split candidate chunks if needed.
`MAX_JOBS=2` performs a smoke check; change to 0 and rerun this cell to finish.
Keep the same settings on all partitions. Smaller OOM subchunks can change scores;
compare experimental results accordingly, not as a bit-identical reproduction.


In [ ]:
def valid_score(value):
    return isinstance(value,(int,float)) and not isinstance(value,bool) and np.isfinite(value) and 0<=value<=1

jobs=[]
for user in selected_users:
    profile,candidates=requests[user]
    saved=cache.get(str(user),{})
    # Preserve original chunk boundaries across resumptions.
    for job in bounded_jobs(user,profile,candidates):
        if not all(valid_score(saved.get(str(i))) for i,_ in job[2]):jobs.append(job)
jobs.sort(key=lambda job:prompt_tokens(job[1],job[2]))
print('Pending chunks:',len(jobs),'Maximum batch:',BATCH_SIZE)
if MAX_JOBS:jobs=jobs[:MAX_JOBS]
fallback_file=OUT_DIR/f'llm_cache_part_{PART_ID:02d}_fallbacks.json'
fallbacks=json.loads(fallback_file.read_text()) if fallback_file.exists() else {}
torch.cuda.reset_peak_memory_stats()
for start in tqdm(range(0,len(jobs),BATCH_SIZE),desc=f'Part {PART_ID}'):
    for user,scores,neutral_ids in adaptive_results(jobs[start:start+BATCH_SIZE]):
        # Never overwrite already valid scores during resume.
        dest=cache.setdefault(str(user),{})
        new_fallbacks=[]
        for item,value in scores.items():
            if not valid_score(dest.get(str(item))):
                dest[str(item)]=float(value)
                if item in neutral_ids:new_fallbacks.append(item)
        fallbacks[str(user)]=sorted(set(fallbacks.get(str(user),[]))|set(new_fallbacks))
        # Diagnostics first: an interruption may overcount fallbacks, never hide them.
        atomic_write(fallback_file,fallbacks)
        atomic_write(cache_file,cache)
print('Saved:',cache_file)
print('Peak allocated GPU GiB:',round(torch.cuda.max_memory_allocated()/2**30,3))
print('Neutral fallback item count:',sum(map(len,fallbacks.values())))
if MAX_JOBS:print('Smoke limit active. Set MAX_JOBS=0 and rerun this cell to finish.')


## Final validation
An incomplete cache is reported clearly; no missing-user KeyError and no false SUCCESS.
A complete cache can contain neutral fallback scores: inspect the fallback count/file
before claiming generated-score quality. Include this diagnostic file when sharing results.


In [ ]:
incomplete=[]
for user in selected_users:
    actual=cache.get(str(user),{})
    expected={str(i) for i,_ in requests[user][1]}
    if set(actual)!=expected or not all(valid_score(v) for v in actual.values()):
        incomplete.append((user,len(expected-set(actual))))
if incomplete:
    print('INCOMPLETE:',len(incomplete),'users. First entries (user, missing count):',incomplete[:10])
    print('Set MAX_JOBS=0 and rerun the scoring cell. Do not merge this part yet.')
else:
    print('COMPLETE:',len(selected_users),'users;',sum(map(len,cache.values())),'item scores')
    print('This validates coverage/ranges, not prediction quality. Inspect neutral fallbacks.')
print('Cache:',cache_file)
print('Manifest:',manifest_file)


## Combining results
Run the **separate merge notebook** once all ten accounts have completed. Collect both `.json` files from each account in a shared folder. The merge notebook validates consistent settings, complete coverage, no duplicated users, and valid score ranges before writing `llm_cache.json`.